# Implementing WCST wtih functioning agent

In [ ]:
"""Data hierarchy for the WCST domain."""

from pyClarion import * # Importing everything from PyClarion
from pyClarion.knowledge import * # Importing all knowledege store-related functions from PyClarion

class Color(Atoms): 
    """A sort for atomic color terms."""
    red: Atom
    grn: Atom
    blu: Atom

class Shape(Atoms): 
    """A sort for atomic shape terms."""
    circ: Atom
    squr: Atom
    tria: Atom

class Number(Atoms):
    """A sort for atomic number terms."""
    one: Atom
    two: Atom
    three: Atom
    four: Atom

class Main(Buses):
    """A sort for main data buses."""
    input: Bus
    output: Bus
    target: Bus


class WCSTBuses(BusFamily):
    """A family for all bus sorts."""
    main: Main


class WCSTData(DataFamily):
    """A family for all data sorts."""
    color: Color
    shape: Shape
    number: Number
    


class WCSTRoot(Root):
    """The root of the model keyspace."""
    b: WCSTBuses
    d: WCSTData


In [ ]:
# Step 2 : Building Agent

class BottomUpAgent[R: Root, D: DVPairs](Agent):
    root: R
    ipt: Input
    chunks: ChunkStore[D]
    bu: BottomUp[D]

    def __init__(self, name: str, root: R, f: DataFamily, d: D) -> None:
        
        super().__init__(name, root)

        # Convenient handle for keyspace root.
        self.root = root
        
        # Any process objects initialized within this block will automatically 
        # be added to the agent's system.
        with self:
            # Create an input process to pass in feature activations. The tuple 
            # `(b, d)`indicates that this process will receive inputs in the 
            # form of dimension-value pairs constructed by pairing symbols from 
            # the families 'b'  and 'd'.
            self.ipt = Input(f"{name}.ipt", d)

            # Create a chunk store using the family 'd' to house chunk symbols 
            # (arg 'c')
            self.chunks = ChunkStore(f"{name}.chunks", c=f, d=d)

            # Create a bottom up activation process dependent on chunk store.
            self.bu = self.ipt >> self.chunks.bottom_up(f"{name}.bu")

# Initialize basic data symbols for current simulation
# During initialization, WCSTData() will automatically generate symbols 
# for all terms annotated with a sort or term type.
root = WCSTRoot()

# Create a new bottom-up demo agent and populate its keyspace with data symbols.
agent = BottomUpAgent("agent", root, root.d, (root.b.main, root.d))

In [ ]:
# Step 3 : Initialize the knowledge of the model's availabe cards

"""Chunks for bottom-up activation model"""

# Define short handles for data sorts
main = agent.root.b.main
color = agent.root.d.color   
shape = agent.root.d.shape  
number = agent.root.d.number 

# Define three chunks. Technically, these chunks are represented as terms.
# Note that dimension-value pairs are constructed by pairing together two terms 
# using the '**' operator. Positive sign indicates a top-down weight of +1.0. 
# Other weights may be assigned by left multiplication with floats.
chunk_defs = [
    # The caret operator can be used to name chunks and rules. 
    "one_blue_triangle" ^ 
    + main.input ** color.blu
    + main.input ** shape.tria
    + main.input ** number.one,

    "one_red_triangle" ^
    + main.input ** color.red
    + main.input ** shape.tria
    + main.input ** number.one,

    "two_green_squares" ^
    + main.input ** color.grn
    + main.input ** shape.squr
    + main.input ** number.two,
]

# Schedule an event to populate the model with the new chunks. This event will 
# bind the terms defined above to the model's keyspace and initialize their 
# top-down and bottom-up weights. The knowledge defined in `chunk_defs` will not 
# be available to the model until this event is executed (during event 
# processing).
agent.system.schedule(agent.chunks.encode(*chunk_defs))

In [ ]:
# Step 4 : 